# Time series 7: Prophet

Fit **Prophet** per frequency band on train data (trend + daily/weekly seasonality), then predict the **test period** (last 8760 hours). Same evaluation: MAE and MASE vs naive.

Note: Prophet is fit **once** per band on train; we predict the whole test window in one call. So forecasts are 1-step, 2-step, …, N-step from the end of train (not rolling 1-step like ARIMA). Can be slow with many bands.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
from prophet import Prophet

_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

freq_bands = sorted([c for c in df.columns if c not in ("date", "hour", "datetime")])
TEST_STEPS = 24 * 365

print(f"Loaded {len(df)} rows. Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Frequency bands: {len(freq_bands)}")
print(f"Test window: last {TEST_STEPS} hours")

Loaded 9192 rows. Date range: 2025-01-01 to 2026-01-18
Frequency bands: 70
Test window: last 8760 hours


## Fit Prophet per band, predict test period

Prophet expects columns `ds` (datetime) and `y` (value). We fit on train, then create a future dataframe with test timestamps (hourly) and predict. Daily and weekly seasonality are enabled by default.

## Tune Prophet hyperparameters (first band, validation)

Grid-search **changepoint_prior_scale** and **seasonality_prior_scale** on the first frequency band using the last 14 days of train as validation. The best config is then used for all bands in the fit cell below. Run this cell first to set `PROPHET_KWARGS`; if skipped, defaults are used.

In [8]:
VAL_HOURS = 24 * 14  # 14 days validation (last 14 days of train)
tune_band = freq_bands[0]
sub = df[["datetime", tune_band]].dropna(subset=[tune_band]).sort_values("datetime").reset_index(drop=True)
if len(sub) < TEST_STEPS + VAL_HOURS + 100:
    PROPHET_KWARGS = dict(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False, interval_width=0)
    print("Not enough data for tuning; using default Prophet kwargs.")
else:
    train_full = sub.iloc[:-TEST_STEPS][["datetime", tune_band]].copy()
    train_full.columns = ["ds", "y"]
    train_fit = train_full.iloc[:-VAL_HOURS]
    train_val = train_full.iloc[-VAL_HOURS:]
    best_mae, best_kw = np.inf, None
    for cp in [0.01, 0.05, 0.1]:
        for sp in [0.1, 1.0, 10.0]:
            try:
                m = Prophet(
                    daily_seasonality=True,
                    weekly_seasonality=True,
                    yearly_seasonality=False,
                    interval_width=0,
                    changepoint_prior_scale=cp,
                    seasonality_prior_scale=sp,
                )
                m.fit(train_fit)
                fcst = m.predict(pd.DataFrame({"ds": train_val["ds"]}))
                mae = float(np.mean(np.abs(train_val["y"].values - fcst["yhat"].values)))
                if mae < best_mae:
                    best_mae, best_kw = mae, {"changepoint_prior_scale": cp, "seasonality_prior_scale": sp}
            except Exception:
                continue
    PROPHET_KWARGS = dict(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
        interval_width=0,
        **(best_kw or {}),
    )
    print(f"Tuned on band {tune_band} (val MAE = {best_mae:.4f}): {PROPHET_KWARGS}")

Not enough data for tuning; using default Prophet kwargs.


In [9]:
rows_prophet = []
n_bands = len(freq_bands)
for bi, freq in enumerate(freq_bands):
    print(f"[Prophet] Band {bi+1}/{n_bands}: {freq} ...", flush=True)
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 100:
        print(f"  Skip: not enough data", flush=True)
        continue
    train_df = sub.iloc[:-TEST_STEPS][["datetime", freq]].copy()
    train_df.columns = ["ds", "y"]
    test_df = sub.iloc[-TEST_STEPS:][["datetime", freq]].copy()
    test_df.columns = ["datetime", "actual"]
    try:
        try:
            kwargs = PROPHET_KWARGS
        except NameError:
            kwargs = dict(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False, interval_width=0)
        m = Prophet(**kwargs)
        m.fit(train_df)
    except Exception as e:
        print(f"  Skip: fit failed ({e})", flush=True)
        continue
    future = pd.DataFrame({"ds": test_df["datetime"]})
    try:
        fcst = m.predict(future)
    except Exception as e:
        print(f"  Skip: predict failed ({e})", flush=True)
        continue
    for i in range(len(test_df)):
        rows_prophet.append({
            "datetime": test_df["datetime"].iloc[i],
            "frequency_band": freq,
            "actual": float(test_df["actual"].iloc[i]),
            "predicted": float(fcst["yhat"].iloc[i]),
        })
    print(f"  Done.", flush=True)

df_prophet = pd.DataFrame(rows_prophet)
print(f"[Prophet] Total: {len(df_prophet)} predictions")

[Prophet] Band 1/70: 3.100-3.105 ...


23:34:18 - cmdstanpy - INFO - Chain [1] start processing
23:34:18 - cmdstanpy - INFO - Chain [1] done processing


  Done.
[Prophet] Band 2/70: 3.105-3.110 ...


23:34:19 - cmdstanpy - INFO - Chain [1] start processing
23:34:19 - cmdstanpy - INFO - Chain [1] done processing


  Done.
[Prophet] Band 3/70: 3.110-3.115 ...


...

  Done.
[Prophet] Band 69/70: 3.440-3.445 ...


23:35:21 - cmdstanpy - INFO - Chain [1] start processing
23:35:21 - cmdstanpy - INFO - Chain [1] done processing


  Done.
[Prophet] Band 70/70: 3.445-3.450 ...


23:35:22 - cmdstanpy - INFO - Chain [1] start processing
23:35:22 - cmdstanpy - INFO - Chain [1] done processing


  Done.
[Prophet] Total: 613200 predictions


## Naive baseline and MASE

In [12]:
rows_naive = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
df_naive = pd.DataFrame(rows_naive)

def mase_from_df(pred_df):
    diffs = pred_df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(pred_df["actual"] - pred_df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

mae_naive = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
mase_naive = mase_from_df(df_naive)
print(f"Naive:  MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"Naive:  MAE = {mae_naive:.4f}")

Naive:  MAE = 2.4198,  MASE = 0.9999


## Prophet vs Naive

In [13]:
if len(df_prophet) == 0:
    print("No Prophet predictions (fit/predict failed for all bands). Check data and Prophet install.")
else:
    mae_prophet = float(np.mean(np.abs(df_prophet["actual"] - df_prophet["predicted"])))
    mase_prophet = mase_from_df(df_prophet)
    print("=== Prophet vs Naive (same test period) ===")
    print(f"  Naive:   MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"  Naive:   MAE = {mae_naive:.4f}")
    print(f"  Prophet: MAE = {mae_prophet:.4f},  MASE = {mase_prophet:.4f}" if not np.isnan(mase_prophet) else f"  Prophet: MAE = {mae_prophet:.4f}")
    if mae_prophet < mae_naive:
        print("  → Prophet beats naive.")
    else:
        print("  → Naive best (common when persistence dominates at 1-step).")

=== Prophet vs Naive (same test period) ===
  Naive:   MAE = 2.4198,  MASE = 0.9999
  Prophet: MAE = 48.1280,  MASE = 19.8891
  → Naive best (common when persistence dominates at 1-step).
